In [ ]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-10-27_00-04-26/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

In [ ]:
# Check the mission log against the dataset just built: which entries have labelled attack
# connections in it. Run after the build cell (uses `spark` and `full` from that cell).
import ipaddress
import pandas as pd
from IPython.display import display

mission_log_path = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/attack_log/"
                    "attack_82913-CNT4416-202608-attack.csv")
PAD = pd.Timedelta(minutes=2)          # the correlation's +/- 2 minute window

# 1. Mission log (every column as text). Line numbers match the CSV file, header = line 1.
log = (spark.read.option("header", True).option("multiline", True).option("encoding", "UTF-8")
       .csv(mission_log_path).toPandas().fillna(""))
log.insert(0, "log_line", range(2, 2 + len(log)))

# 2. Attack connections summarized per source IP, destination IP, T code and day
summary = (full.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
               .withColumn("day", F.substring("datetime", 1, 10))
               .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "day")
               .agg(F.count(F.lit(1)).alias("connections"),
                    F.min("datetime").alias("first"), F.max("datetime").alias("last"))
               .toPandas())
summary["t0"] = pd.to_datetime(summary["first"], utc=True)
summary["t1"] = pd.to_datetime(summary["last"], utc=True)
summary["matched"] = False

def network(text):
    try:
        return ipaddress.ip_network(str(text).strip(), strict=False)
    except ValueError:
        return None

def third_octet(text):
    try:
        return int(str(text).split("/")[0].split(".")[2])
    except (IndexError, ValueError):
        return None

def single_port(text):
    t = str(text).split("/")[0].strip()
    return int(t) if t.isdigit() else None

# 3. Match each entry: same T code, IPs inside the entry's IPs (or /24), time within start-2min..end+2min
rows = []
for _, e in log.iterrows():
    src, dst = network(e.src_ip), network(e.dest_ip)
    start = pd.to_datetime(e.datetime_start, utc=True, errors="coerce")
    end = pd.to_datetime(e.datetime_end, utc=True, errors="coerce")

    notes = []
    if e.mitre_attack_technique == "T0000":
        notes.append("benign window marker (T0000), not an attack")
    if pd.notna(start) and pd.notna(end) and end < start:
        notes.append("end time before start time")
    if "/" in e.src_ip:
        notes.append("source entered as a range")
    if src is not None and dst is not None and src == dst and src.num_addresses == 1:
        notes.append("source and destination are the same host (not visible on the network)")
    port = single_port(e.dest_port)
    if port is not None and port >= 32768:
        notes.append("dest port looks like an ephemeral/RHP port")
    if src is None or dst is None or pd.isna(start) or pd.isna(end):
        notes.append("could not read IP or time")

    m = summary.iloc[0:0]
    if src is not None and dst is not None and pd.notna(start) and pd.notna(end):
        m = summary[(summary.label_technique == e.mitre_attack_technique)
                    & summary.src_ip_zeek.map(lambda ip: ipaddress.ip_address(ip) in src)
                    & summary.dest_ip_zeek.map(lambda ip: ipaddress.ip_address(ip) in dst)
                    & (summary.t0 - PAD <= end) & (summary.t1 + PAD >= start)]
        summary.loc[m.index, "matched"] = True

    rows.append({
        "Log line": e.log_line, "Group": e.group_id, "T code": e.mitre_attack_technique,
        "Source IP": e.src_ip, "Dest IP": e.dest_ip, "Dest port": e.dest_port,
        "Start (UTC)": e.datetime_start, "End (UTC)": e.datetime_end,
        "Source subnet": third_octet(e.src_ip), "Target subnet": third_octet(e.dest_ip),
        "In dataset": "N/A" if e.mitre_attack_technique == "T0000" else ("Yes" if len(m) else "No"),
        "Labelled connections": int(m.connections.sum()),
        "Notes": "; ".join(notes)})
results = pd.DataFrame(rows)

# 4. Output
attacks = results[results["In dataset"] != "N/A"]
yes = lambda x: int((x["In dataset"] == "Yes").sum())
print(f"Entries in dataset: {yes(attacks)} of {len(attacks)} attack entries\n")

nets = sorted(set(attacks["Target subnet"].dropna()) | set(attacks["Source subnet"].dropna()))
tally = []
for n in nets:
    recv = attacks[attacks["Target subnet"] == n]
    self_scans = recv[recv["Source subnet"] == n]
    made = attacks[(attacks["Source subnet"] == n) & (attacks["Target subnet"] != n)]
    tally.append({"Subnet": f"143.88.{int(n)}.x",
                  "Attacks on it (in/total)": f"{yes(recv)}/{len(recv)}",
                  "of which self-scans": f"{yes(self_scans)}/{len(self_scans)}",
                  "Attacks made from it (in/total)": f"{yes(made)}/{len(made)}"})
print("Per subnet (attacks on it = DCO view; OCO = attacks on the subnet a group targeted):")
display(pd.DataFrame(tally))

problems = results[(results["Notes"] != "") & (results["In dataset"] == "No")]
print("Missing entries with an obvious problem in the log:")
display(problems[["Log line", "Group", "T code", "Source IP", "Dest IP", "Dest port", "Notes"]]
        if len(problems) else "none")

unmatched = summary[~summary["matched"]]
print(f"Labelled connection groups that matched no entry: {len(unmatched)} (should be 0)")
if len(unmatched):
    display(unmatched[["src_ip_zeek", "dest_ip_zeek", "label_technique", "first", "connections"]])

print("All entries:")
display(results)

In [ ]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#

In [1]:
#test missing entries
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "6g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

logs = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/nsm/zeek/logs"

checks = [
    (4,  "143.88.2.13",  "143.88.2.1",    "2026-09-01T20:56:01Z", "2026-09-01T20:56:06Z"),  # G2 T1595 443
    (6,  "143.88.2.13",  "143.88.2.11",   "2026-09-01T20:57:30Z", "2026-09-01T20:57:35Z"),  # G2 T1595 80
    (7,  "143.88.2.13",  "143.88.2.12",   "2026-09-01T20:57:38Z", "2026-09-01T20:57:40Z"),  # G2 T1595 80
    (8,  "143.88.2.13",  "143.88.2.13",   "2026-09-01T20:57:48Z", "2026-09-01T20:57:50Z"),  # G2 T1595 8000 (dst = src)
    (16, "143.88.4.13",  "143.88.255.11", "2026-08-30T15:22:01Z", "2026-08-30T15:22:09Z"),  # G4 T1595 80
    (28, "143.88.6.13",  "143.88.6.12",   "2026-09-08T15:15:36Z", "2026-09-08T15:15:42Z"),  # G5 T1046 (src is a group 6 IP)
    (35, "143.88.8.10",  "143.88.9.10",   "2026-09-09T19:41:05Z", "2026-09-09T19:41:20Z"),  # G8 T1595
    (36, "143.88.6.10",  "143.88.7.",     "2026-09-14T14:59:00Z", "2026-09-14T15:01:00Z"),  # G6 T1595
    (47, "143.88.9.13",  "143.88.10.12",  "2026-09-17T00:10:22Z", "2026-09-17T00:10:22Z"),  # G9 T1210 445
    (54, "143.88.1.11",  "143.88.10.10",  "2026-09-22T13:50:00Z", "2026-09-22T15:05:00Z"),  # G10 T1564 4446
    (55, "143.88.10.10", "143.88.1.18",   "2026-09-22T13:50:00Z", "2026-09-22T14:29:00Z"),  # G10 T1070 4444
    (56, "143.88.4.13",  "143.88.5.12",   "2026-09-22T20:56:41Z", "2026-09-22T20:56:57Z"),  # G4 T1134 445
    (57, "143.88.9.10",  "143.88.10.12",  "2026-09-22T17:37:53Z", "2026-09-22T18:15:00Z"),  # G9 T1210 445
    (58, "143.88.9.10",  "143.88.10.11",  "2026-09-22T18:18:26Z", "2026-09-22T18:25:00Z"),  # G9 T1070 21
    (60, "143.88.4.10",  "143.88.4.12",   "2026-09-23T13:59:47Z", "2026-09-23T14:00:17Z"),  # G3 T1210 445 (pivot)
    (63, "143.88.2.10",  "143.88.3.12",   "2026-09-23T21:23:00Z", "2026-09-23T21:41:00Z"),  # G2 T1070 445
    (66, "143.88.1.17",  "143.88.2.12",   "2026-09-24T03:44:47Z", "2026-09-24T04:35:34Z"),  # G1 T1070
    (67, "143.88.7.13",  "143.88.8.11",   "2026-09-23T23:39:00Z", "2026-09-23T23:40:00Z"),  # G7 T1070 21
]
def epoch(s): return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()

# Only the day folders that exist for these entries (the day before through the day after)
days = sorted({(datetime.fromisoformat(s[:10]) + timedelta(days=k)).strftime("%Y-%m-%d")
               for _, _, _, s, _ in checks for k in (-1, 0, 1)})
jvm, conf = spark._jvm, spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.Path(logs).getFileSystem(conf)
paths = [f"{logs}/{d}/conn.*.log.gz" for d in days
         if fs.globStatus(jvm.org.apache.hadoop.fs.Path(f"{logs}/{d}/conn.*.log.gz"))]
print("Reading", len(paths), "day folders")

conn = (spark.read.json(paths)
        .select(F.col("`id.orig_h`").alias("src"), F.col("`id.resp_h`").alias("dst"),
                F.col("`id.resp_p`").alias("dport"),
                F.coalesce(F.col("ts").cast("double"), F.to_timestamp("ts").cast("double")).alias("ts"))
        .cache())

windows = [("as logged (±2 min)", 0, 120), ("+5 h (local as UTC)", 5 * 3600, 120),
           ("whole day (±12 h)", 0, 12 * 3600)]
for line, src, dst, start, end in checks:
    dmatch = F.col("dst").startswith(dst) if dst.endswith(".") else (F.col("dst") == dst)
    pair = conn.filter((F.col("src") == src) & dmatch)
    print(f"\nLine {line}: {src} -> {dst}  logged {start}")
    for name, shift, pad in windows:
        r = (pair.filter(F.col("ts").between(epoch(start) + shift - pad, epoch(end) + shift + pad))
                 .agg(F.count(F.lit(1)).alias("n"),
                      F.array_sort(F.collect_set("dport")).alias("ports"),
                      F.from_unixtime(F.min("ts").cast("long")).alias("first"))
                 .first())
        ports = ", ".join(str(p) for p in (r["ports"] or [])[:8])
        print(f"  {name:22s} {r['n']:>6} conns  ports={ports}  first={r['first']}")
    any_src = conn.filter((F.col("src") == src)
                          & F.col("ts").between(epoch(start) - 43200, epoch(end) + 43200)).count()
    print(f"  {'any traffic from src':22s} {any_src:>6} conns that day (to anywhere)")

26/10/26 17:32:02 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/26 17:32:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 17:32:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 17:32:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/26 17:32:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Reading 18 day folders



Line 4: 143.88.2.13 -> 143.88.2.1  logged 2026-09-01T20:56:01Z


  as logged (±2 min)        160 conns  ports=53  first=2026-09-01 20:54:17
  +5 h (local as UTC)      2164 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-02 01:54:23
  whole day (±12 h)       57849 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-01 08:56:01
  any traffic from src    61999 conns that day (to anywhere)

Line 6: 143.88.2.13 -> 143.88.2.11  logged 2026-09-01T20:57:30Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)      1992 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-02 01:57:13
  whole day (±12 h)        1992 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-02 01:57:13
  any traffic from src    61999 conns that day (to anywhere)

Line 7: 143.88.2.13 -> 143.88.2.12  logged 2026-09-01T20:57:38Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)      1155 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-02 01:57:36
  whole day (±12 h)        1155 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  f